In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import expr

spark = SparkSession.builder.appName("SaltingExample").getOrCreate()



In [0]:
# Example large table (orders)
orders = spark.createDataFrame(
    [(1, "A"), (1, "B"), (1, "C"), (2, "D"), (3, "E")],
    ["customer_id", "order"]
)

# Example small table (customers)
customers = spark.createDataFrame(
    [(1, "Alice"), (2, "Bob"), (3, "Charlie")],
    ["customer_id", "name"]
)



In [0]:
# Step 1: Add random salt to the large table
orders_salted = orders.withColumn("salt", expr("floor(rand() * 5)"))



In [0]:
# Step 2: Create salt range (0 to 4)
salt_range = spark.range(0, 5).withColumnRenamed("id", "salt")



In [0]:
# Step 3: Duplicate small table with all salt values
customers_salted = customers.crossJoin(salt_range)



In [0]:
# Step 4: Perform salted join
joined = orders_salted.join(
    customers_salted,
    (orders_salted.customer_id == customers_salted.customer_id) &
    (orders_salted.salt == customers_salted.salt),
    "inner"
)

# Step 5: Drop salt column if not needed
result = joined.drop("salt")

result.show()


+-----------+-----+-----------+-------+
|customer_id|order|customer_id|   name|
+-----------+-----+-----------+-------+
|          1|    B|          1|  Alice|
|          1|    C|          1|  Alice|
|          1|    A|          1|  Alice|
|          2|    D|          2|    Bob|
|          3|    E|          3|Charlie|
+-----------+-----+-----------+-------+

